# MNIST 手書き数字認識 — 畳み込みニューラルネットワーク (CNN)

MLP では画像を1次元に展開 (flatten) して入力していたが、CNN は **画像の空間構造 (2D)** をそのまま活かして特徴を抽出する。

## MLP との違い

| | MLP | CNN |
|---|---|---|
| 入力 | 784次元ベクトル (flatten) | 1×28×28 の画像そのまま |
| 特徴抽出 | 全結合層 (全ピクセル同時に処理) | フィルタが画像上をスライド |
| パラメータ数 | 多い (全ピクセル×全ニューロン) | 少ない (フィルタを共有) |
| 位置の扱い | 位置情報が失われる | 近傍の関係を保持 |

## 全体の流れ
1. セットアップ
2. データの準備
3. 畳み込みの仕組みを理解する
4. CNN モデルの定義
5. 学習
6. 評価
7. 特徴マップの可視化
8. MLP との比較

## 1. セットアップ

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# デバイス設定
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"使用デバイス: {device}")

## 2. データの準備

MLP のときと同じ MNIST データセットを使う。
ただし CNN では flatten **しない** — `(batch, 1, 28, 28)` の形状のまま入力する。

In [ ]:
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 10

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 入力の形状を確認 — CNN は (batch, channels, height, width) で受け取る
sample, _ = train_dataset[0]
print(f"入力テンソルの形状: {sample.shape}")  # (1, 28, 28)
print(f"訓練データ: {len(train_dataset)} 枚 / テストデータ: {len(test_dataset)} 枚")

## 3. 畳み込みの仕組みを理解する

CNN の中核は **畳み込み層 (Conv2d)** と **プーリング層 (MaxPool2d)**。
実際にデータを通して、各操作で何が起きるか確認する。

### 畳み込み (Convolution) とは

小さなフィルタ (カーネル) を画像上でスライドさせ、局所的な特徴を検出する。

```
入力画像 (5×5)          フィルタ (3×3)        出力 (3×3)
┌─────────────┐      ┌─────────┐      ┌─────────┐
│ 1 0 1 0 1   │      │ 1 0 1   │      │ ? ? ?   │
│ 0 1 0 1 0   │  ×   │ 0 1 0   │  →   │ ? ? ?   │
│ 1 0 1 0 1   │      │ 1 0 1   │      │ ? ? ?   │
│ 0 1 0 1 0   │      └─────────┘      └─────────┘
│ 1 0 1 0 1   │
└─────────────┘
```

フィルタが画像上を1ピクセルずつスライドし、重なった部分の要素積の和を計算する。
- **フィルタの値は学習で自動的に決まる** (エッジ検出、角検出など)
- 複数のフィルタを使えば、複数の特徴を同時に検出できる

### 出力サイズの計算

```
出力サイズ = (入力サイズ - カーネルサイズ + 2×パディング) / ストライド + 1
```

例: 入力 28×28, カーネル 3×3, パディング 1, ストライド 1 → 出力 28×28 (サイズ維持)

In [ ]:
# 畳み込みの動作を実際に確認する
sample_image, sample_label = train_dataset[0]
print(f"入力: {sample_image.shape}")  # (1, 28, 28)

# Conv2d(入力チャネル, 出力チャネル, カーネルサイズ)
conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
pool = nn.MaxPool2d(2, 2)

# 畳み込みを適用 (バッチ次元を追加)
x = sample_image.unsqueeze(0)  # (1, 1, 28, 28)
print(f"バッチ追加後: {x.shape}")

x_conv = conv1(x)
print(f"畳み込み後:   {x_conv.shape}")  # (1, 16, 28, 28) — 16枚の特徴マップ

x_pool = pool(x_conv)
print(f"プーリング後: {x_pool.shape}")  # (1, 16, 14, 14) — サイズ半分

### 各操作の効果を可視化

同じ画像に対して、畳み込み → ReLU → プーリングの各段階を見てみる。

In [ ]:
with torch.no_grad():
    x = sample_image.unsqueeze(0)
    x_conv = conv1(x)
    x_relu = F.relu(x_conv)
    x_pool = pool(x_relu)

fig, axes = plt.subplots(4, 5, figsize=(14, 11))

# 元画像
axes[0, 0].imshow(sample_image.squeeze(), cmap="gray")
axes[0, 0].set_title(f"元画像 (ラベル: {sample_label})")
for j in range(1, 5):
    axes[0, j].axis("off")

# 畳み込み後 (最初の4チャネル)
for j in range(5):
    if j == 0:
        axes[1, j].set_ylabel("Conv2d", fontsize=12)
    axes[1, j].imshow(x_conv[0, j].numpy(), cmap="viridis")
    axes[1, j].set_title(f"フィルタ {j}")

# ReLU 後
for j in range(5):
    if j == 0:
        axes[2, j].set_ylabel("ReLU", fontsize=12)
    axes[2, j].imshow(x_relu[0, j].numpy(), cmap="viridis")
    axes[2, j].set_title(f"フィルタ {j}")

# プーリング後
for j in range(5):
    if j == 0:
        axes[3, j].set_ylabel("MaxPool", fontsize=12)
    axes[3, j].imshow(x_pool[0, j].numpy(), cmap="viridis")
    axes[3, j].set_title(f"フィルタ {j}")

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("畳み込み → ReLU → プーリング の各段階", fontsize=14)
plt.tight_layout()
plt.show()

### プーリング (Pooling) とは

特徴マップのサイズを縮小し、位置の微小なズレに頑健にする。

```
MaxPool2d(2, 2) の動作:

┌───┬───┐         ┌───┐
│ 1 │ 3 │         │   │
├───┼───┤  → max  │ 4 │
│ 2 │ 4 │         │   │
└───┴───┘         └───┘

2×2 の領域から最大値を取る → サイズが半分になる
```

### CNN の全体像

```
入力画像 → [Conv → ReLU → Pool] × N → Flatten → 全結合層 → 出力
          ~~~~~~~~~~~~~~~~~~~~~~~~     ~~~~~~~~~~~~~~~~~~~~
          特徴抽出 (空間構造を活かす)    分類 (MLP と同じ)
```

CNN は「特徴抽出器 (畳み込み部分) + 分類器 (全結合部分)」の組み合わせ。

## 4. CNN モデルの定義

```
入力 (1, 28, 28)
  ↓ Conv2d(1→16, 3×3, padding=1)  → (16, 28, 28)
  ↓ ReLU
  ↓ MaxPool2d(2×2)                → (16, 14, 14)
  ↓ Conv2d(16→32, 3×3, padding=1) → (32, 14, 14)
  ↓ ReLU
  ↓ MaxPool2d(2×2)                → (32, 7, 7)
  ↓ Flatten                        → (32×7×7 = 1568)
  ↓ Linear(1568→128)
  ↓ ReLU
  ↓ Linear(128→10)
出力 (10)
```

### パラメータ数の比較

MLP (784→256→128→10) のパラメータ数: 約 235,000

CNN の畳み込み層のパラメータ数:
- Conv1: `1×16×3×3 + 16(bias)` = 160
- Conv2: `16×32×3×3 + 32(bias)` = 4,640

**フィルタを画像全体で共有するため、畳み込み部分のパラメータは非常に少ない。**

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 特徴抽出部分
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # (1,28,28) → (16,28,28)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # (16,28,28) → (16,14,14)
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # (16,14,14) → (32,14,14)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # (32,14,14) → (32,7,7)
        )
        # 分類部分
        self.classifier = nn.Sequential(
            nn.Flatten(),                                 # (32,7,7) → (1568)
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CNN().to(device)
print(model)

# パラメータ数を確認
total_params = sum(p.numel() for p in model.parameters())
conv_params = sum(p.numel() for p in model.features.parameters())
fc_params = sum(p.numel() for p in model.classifier.parameters())
print(f"\n総パラメータ数:     {total_params:,}")
print(f"  畳み込み部分:     {conv_params:,}")
print(f"  全結合部分:       {fc_params:,}")

## 5. 学習

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_losses = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch [{epoch+1}/{EPOCHS}]  Loss: {avg_loss:.4f}")

print("\n学習完了!")

## 6. 評価

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"テストデータでの正解率: {accuracy:.2f}% ({correct}/{total})")

In [ ]:
# 学習曲線
plt.figure(figsize=(8, 5))
plt.plot(range(1, EPOCHS + 1), train_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("学習曲線 (Training Loss)")
plt.grid(True)
plt.show()

In [ ]:
# 予測結果の可視化
model.eval()
fig, axes = plt.subplots(3, 5, figsize=(14, 9))

with torch.no_grad():
    for i, ax in enumerate(axes.flat):
        image, label = test_dataset[i]
        output = model(image.unsqueeze(0).to(device))
        pred = output.argmax(dim=1).item()

        ax.imshow(image.squeeze(), cmap="gray")
        color = "green" if pred == label else "red"
        ax.set_title(f"予測: {pred} (正解: {label})", color=color, fontsize=11)
        ax.axis("off")

plt.suptitle("テストデータに対する予測結果", fontsize=14)
plt.tight_layout()
plt.show()

## 7. 学習済みフィルタと特徴マップの可視化

CNN が学習した **フィルタ (重み)** と、それを画像に適用した結果 **(特徴マップ)** を見てみる。

- フィルタ: CNN が「何を見ようとしているか」
- 特徴マップ: 実際の画像で「何が検出されたか」

In [ ]:
# 第1層の学習済みフィルタを可視化
conv1_weights = model.features[0].weight.data.cpu()  # (16, 1, 3, 3)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(conv1_weights[i, 0], cmap="RdBu_r", vmin=-0.5, vmax=0.5)
    ax.set_title(f"フィルタ {i}", fontsize=9)
    ax.axis("off")

plt.suptitle("第1畳み込み層の学習済みフィルタ (3×3)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# テスト画像に対する特徴マップを可視化
model.eval()
sample_image, sample_label = test_dataset[0]

with torch.no_grad():
    x = sample_image.unsqueeze(0).to(device)

    # 各層の出力を手動で取得
    x1 = model.features[0](x)   # Conv1
    x1r = model.features[1](x1) # ReLU
    x1p = model.features[2](x1r) # Pool
    x2 = model.features[3](x1p) # Conv2
    x2r = model.features[4](x2) # ReLU

fig, axes = plt.subplots(3, 9, figsize=(16, 6))

# 元画像
axes[0, 0].imshow(sample_image.squeeze(), cmap="gray")
axes[0, 0].set_title(f"元画像 ({sample_label})", fontsize=10)
for j in range(1, 9):
    axes[0, j].axis("off")

# 第1層の特徴マップ (最初の8チャネル)
for j in range(9):
    if j == 0:
        axes[1, j].set_ylabel("Conv1 出力", fontsize=10)
    if j < 8:
        axes[1, j].imshow(x1r[0, j].cpu().numpy(), cmap="viridis")
        axes[1, j].set_title(f"ch {j}", fontsize=9)
    else:
        axes[1, j].axis("off")

# 第2層の特徴マップ (最初の8チャネル)
for j in range(9):
    if j == 0:
        axes[2, j].set_ylabel("Conv2 出力", fontsize=10)
    if j < 8:
        axes[2, j].imshow(x2r[0, j].cpu().numpy(), cmap="viridis")
        axes[2, j].set_title(f"ch {j}", fontsize=9)
    else:
        axes[2, j].axis("off")

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("各畳み込み層の特徴マップ", fontsize=14)
plt.tight_layout()
plt.show()

## 8. MLP との比較

同じ MNIST データセットで MLP と CNN を同条件で比較する。

In [ ]:
# 共通の学習・評価関数
def train_and_evaluate(model, name):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    losses = []

    print(f"\n--- {name} ---")
    params = sum(p.numel() for p in model.parameters())
    print(f"  パラメータ数: {params:,}")

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        avg_loss = running_loss / len(train_loader)
        losses.append(avg_loss)
        print(f"  Epoch [{epoch+1}/{EPOCHS}]  Loss: {avg_loss:.4f}")

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = 100 * correct / total
    print(f"  テスト正解率: {acc:.2f}%")

    return {"losses": losses, "accuracy": acc, "params": params}

In [ ]:
# MLP (比較用)
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.net(x)

# CNN (上と同じ構成)
class CNN_Compare(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

print("MLP vs CNN 比較実験")
comparison = {}
comparison["MLP (784→256→128→10)"] = train_and_evaluate(MLP(), "MLP")
comparison["CNN (Conv16→Conv32→FC128)"] = train_and_evaluate(CNN_Compare(), "CNN")

In [ ]:
# 結果を可視化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 学習曲線
for name, data in comparison.items():
    ax1.plot(range(1, EPOCHS + 1), data["losses"], marker="o", label=name)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("学習曲線")
ax1.legend()
ax1.grid(True)

# 正解率とパラメータ数
names = list(comparison.keys())
accs = [comparison[n]["accuracy"] for n in names]
params = [comparison[n]["params"] for n in names]
colors = ["#4C72B0", "#DD8452"]
bars = ax2.bar(names, accs, color=colors)
ax2.set_ylabel("正解率 (%)")
ax2.set_title("テスト正解率")
ax2.set_ylim(max(0, min(accs) - 5), 100)
for bar, acc, p in zip(bars, accs, params):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             f"{acc:.1f}%\n({p:,} params)", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

print("\nまとめ:")
for name, data in comparison.items():
    print(f"  {name}: 正解率 {data['accuracy']:.2f}%, パラメータ数 {data['params']:,}")

## まとめ

### CNN が MLP より優れる理由

1. **局所的な特徴の抽出**: フィルタが近傍ピクセルの関係を捉える (エッジ、角、曲線など)
2. **パラメータの共有**: 同じフィルタを画像全体で使い回すため、少ないパラメータで効率的に学習
3. **平行移動への頑健性**: 数字が画像内で少しずれても、同じフィルタで検出できる
4. **階層的な特徴学習**: 浅い層でエッジ → 深い層で形状やパーツ → 最終的に数字全体を認識

### 次のステップ

- **Dropout**: 過学習を防ぐ正則化手法
- **Batch Normalization**: 学習を安定化させるテクニック
- **データ拡張 (Data Augmentation)**: 回転・シフトなどで訓練データを増やす
- **より深い構成**: VGG, ResNet などの有名アーキテクチャへの発展